# Regression Task: Traffic Speed Prediction

## Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import torch
import gpytorch
import networkx as nx
from tqdm.auto import tqdm
from grf_gp.sampler import GRFSampler
from grf_gp.utils.spectral import get_normalized_laplacian
from grf_gp.kernels.base import BaseExactKernel
from grf_gp.kernels.diffusion import DiffusionGRFKernel
from grf_gp.kernels.general import GeneralGRFKernel
from grf_gp.model import GRFGP, ExactGraphGP

import sys
import os
project_root = os.path.abspath("../../..")
sys.path.append(project_root)

from traffic_utils.preprocessing import load_PEMS
from experiments.regression.utils import train_model, evaluate_model
from traffic_utils.plotting import plot_PEMS

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
MAX_WALK_LENGTH = 3
WALKS_PER_NODE = 10000
P_HALT = 0.5
PATHWISE_SAMPLES = 100

In [3]:
# Load and preprocess the PEMS dataset
np.random.seed(1111)
torch.manual_seed(1111)
num_train = 250

G, data_train, data_test, data = load_PEMS(num_train=num_train, path=os.path.join(project_root, "data/traffic"))
x_train, y_train = data_train
x_test, y_test = data_test
x, y = data

orig_mean, orig_std = np.mean(y_train), np.std(y_train)
y_train_norm = (y_train - orig_mean) / orig_std
y_test_norm = (y_test - orig_mean) / orig_std

X_train = torch.tensor(x_train).long().to(device)
Y_train = torch.tensor(y_train_norm).to(device).flatten()
X_test = torch.tensor(x_test).long().to(device)
Y_test = torch.tensor(y_test_norm).to(device).flatten()
X_full = torch.tensor(x).long().to(device)

adjacency_matrix = nx.to_numpy_array(G)
L = get_normalized_laplacian(adjacency_matrix)
L = torch.tensor(L).to(device)


/Users/matthew/Documents/Efficient Gaussian Process on Graphs/Efficient_Gaussian_Process_On_Graphs/experiments/regression/traffic_prediction/traffic_utils/preprocessing.py:113: UserWarning: Unpickling a shapely <2.0 geometry object. Please save the pickle again; shapely 2.1 will not have this compatibility.
  G = pickle.load(f)


epsg:4326


/Users/matthew/Documents/Efficient Gaussian Process on Graphs/Efficient_Gaussian_Process_On_Graphs/venv/lib/python3.11/site-packages/osmnx/convert.py:381: FutureWarning: <class 'geopandas.array.GeometryArray'>._reduce will require a `keepdims` parameter in the future
  dupes = edges[mask].dropna(subset=["geometry"])


## Baseline: Exact Diffusion

In [5]:
class ExactDiffusionKernel(BaseExactKernel):
    def __init__(self, L, beta_max=2.0, **kwargs):
        super().__init__(**kwargs)
        self.register_buffer("L", L)
        self.beta_max = beta_max # prevent oversmoothing

        self.register_parameter(
            name="raw_beta", parameter=torch.nn.Parameter(torch.tensor(1.0))
        )
        self.register_parameter(
            name="raw_sigma_f", parameter=torch.nn.Parameter(torch.tensor(1.0))
        )

    @property
    def beta(self):
        return self.beta_max * torch.sigmoid(self.raw_beta)   # in (0, beta_max)

    @property
    def sigma_f(self):
        return torch.nn.functional.softplus(self.raw_sigma_f)

    def _full_kernel_matrix(self) -> torch.Tensor:
        return self.sigma_f**2 * torch.matrix_exp(-self.beta * self.L)


In [ ]:
likelihood = gpytorch.likelihoods.GaussianLikelihood().to(device)
kernel = ExactDiffusionKernel(L).to(device)
model = ExactGraphGP(X_train, Y_train, likelihood, kernel).to(device)

In [7]:
train_losses = train_model(model, likelihood, X_train, Y_train, lr=0.05, max_iter=100, print_every=1)

Training:   0%|          | 0/100 [00:00<?, ?it/s]

Iter 1/100 - Loss: 1.4179
Iter 2/100 - Loss: 1.4109
Iter 3/100 - Loss: 1.4047
Iter 4/100 - Loss: 1.3995
Iter 5/100 - Loss: 1.3953
Iter 6/100 - Loss: 1.3922
Iter 7/100 - Loss: 1.3902
Iter 8/100 - Loss: 1.3891
Iter 9/100 - Loss: 1.3889
Iter 10/100 - Loss: 1.3893
Iter 11/100 - Loss: 1.3899
Iter 12/100 - Loss: 1.3905
Iter 13/100 - Loss: 1.3909
Iter 14/100 - Loss: 1.3909
Iter 15/100 - Loss: 1.3905
Iter 16/100 - Loss: 1.3899
Iter 17/100 - Loss: 1.3891
Iter 18/100 - Loss: 1.3881
Iter 19/100 - Loss: 1.3872
Iter 20/100 - Loss: 1.3864
Iter 21/100 - Loss: 1.3858
Iter 22/100 - Loss: 1.3853
Iter 23/100 - Loss: 1.3850
Iter 24/100 - Loss: 1.3848
Iter 25/100 - Loss: 1.3847
Iter 26/100 - Loss: 1.3847
Iter 27/100 - Loss: 1.3847
Iter 28/100 - Loss: 1.3846
Iter 29/100 - Loss: 1.3845
Iter 30/100 - Loss: 1.3843
Iter 31/100 - Loss: 1.3841
Iter 32/100 - Loss: 1.3838
Iter 33/100 - Loss: 1.3836
Iter 34/100 - Loss: 1.3833
Iter 35/100 - Loss: 1.3830
Iter 36/100 - Loss: 1.3828
Iter 37/100 - Loss: 1.3826
Iter 38/10

In [8]:
lml, rmse, nlpd = evaluate_model(model, likelihood, X_train, Y_train, X_test, Y_test, orig_std)

/Users/matthew/Documents/Efficient Gaussian Process on Graphs/Efficient_Gaussian_Process_On_Graphs/venv/lib/python3.11/site-packages/gpytorch/models/exact_gp.py:296: GPInputWarning: The input matches the stored training data. Did you forget to call model.train()?
  warnings.warn(


In [9]:
print(f"Log Marginal Likelihood: {lml:.4f}")
print(f"Test RMSE: {rmse:.4f}")
print(f"Test NLPD: {nlpd:.4f}")
print(f"kernel lengthscale: {kernel.beta.item():.4f}")
print(f"kernel stddev: {kernel.sigma_f.item():.4f}")

Log Marginal Likelihood: -208.5882
Test RMSE: 14.8918
Test NLPD: 95.0817
kernel lengthscale: 1.9372
kernel stddev: 1.2899


## Random Walk Sampling

In [10]:
sampler = GRFSampler(L, WALKS_PER_NODE, P_HALT, MAX_WALK_LENGTH, seed=1, use_tqdm=True, n_processes=4)
rw_mats = sampler.sample_random_walk_matrices()

/Users/matthew/Documents/Efficient Gaussian Process on Graphs/Efficient_Gaussian_Process_On_Graphs/venv/lib/python3.11/site-packages/grf_gp/utils/csr.py:17: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/SparseCsrTensorImpl.cpp:55.)
  return adjacency.to_sparse_csr()


## General (Polynomial) GRFs

In [ ]:
likelihood = gpytorch.likelihoods.GaussianLikelihood().to(device)
kernel = GeneralGRFKernel(rw_mats, MAX_WALK_LENGTH).to(device)
model = GRFGP(X_train, Y_train, likelihood, kernel).to(device)

In [12]:
train_losses = train_model(model, likelihood, X_train, Y_train, lr=0.05, max_iter=1000, print_every=1)

Training:   0%|          | 0/1000 [00:00<?, ?it/s]

Iter 1/1000 - Loss: 2.3517
Iter 2/1000 - Loss: 2.3081
Iter 3/1000 - Loss: 2.2629
Iter 4/1000 - Loss: 2.2161
Iter 5/1000 - Loss: 2.1674
Iter 6/1000 - Loss: 2.1168
Iter 7/1000 - Loss: 2.0641
Iter 8/1000 - Loss: 2.0090
Iter 9/1000 - Loss: 1.9515
Iter 10/1000 - Loss: 1.8914
Iter 11/1000 - Loss: 1.8287
Iter 12/1000 - Loss: 1.7637
Iter 13/1000 - Loss: 1.6973
Iter 14/1000 - Loss: 1.6311
Iter 15/1000 - Loss: 1.5677
Iter 16/1000 - Loss: 1.5114
Iter 17/1000 - Loss: 1.4675
Iter 18/1000 - Loss: 1.4407
Iter 19/1000 - Loss: 1.4310
Iter 20/1000 - Loss: 1.4312
Iter 21/1000 - Loss: 1.4315
Iter 22/1000 - Loss: 1.4272
Iter 23/1000 - Loss: 1.4209
Iter 24/1000 - Loss: 1.4182
Iter 25/1000 - Loss: 1.4221
Iter 26/1000 - Loss: 1.4312
Iter 27/1000 - Loss: 1.4419
Iter 28/1000 - Loss: 1.4507
Iter 29/1000 - Loss: 1.4559
Iter 30/1000 - Loss: 1.4568
Iter 31/1000 - Loss: 1.4536
Iter 32/1000 - Loss: 1.4473
Iter 33/1000 - Loss: 1.4393
Iter 34/1000 - Loss: 1.4310
Iter 35/1000 - Loss: 1.4240
Iter 36/1000 - Loss: 1.4192
I

In [13]:
lml, rmse, nlpd = evaluate_model(model, likelihood, X_train, Y_train, X_test, Y_test, orig_std)

/Users/matthew/Documents/Efficient Gaussian Process on Graphs/Efficient_Gaussian_Process_On_Graphs/venv/lib/python3.11/site-packages/linear_operator/utils/sparse.py:51: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  if nonzero_indices.storage():
/Users/matthew/Documents/Efficient Gaussian Process on Graphs/Efficient_Gaussian_Process_On_Graphs/venv/lib/python3.11/site-packages/linear_operator/utils/sparse.py:66: UserWarning: torch.sparse.SparseTensor(indices, values, shape, *, device=) is deprecated.  Please use torch.sparse_coo_tensor(indices, values, shape, dtype=, device=). (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_new.cpp:656.)
  res = cls(index_tensor, value_tensor, interp_size)


In [14]:
print(f"Log Marginal Likelihood: {lml:.4f}")
print(f"Test RMSE: {rmse:.4f}")
print(f"Test NLPD: {nlpd:.4f}")
print(f"kernel modulation: {kernel.modulation_function.detach().cpu().numpy()}")

Log Marginal Likelihood: 195.3118
Test RMSE: 14.0307
Test NLPD: 93.2084
kernel modulation: [ 1.6772591 -3.5183127  1.1930363]


## Diffusion GRFs

In [ ]:
likelihood = gpytorch.likelihoods.GaussianLikelihood().to(device)
kernel = DiffusionGRFKernel(rw_mats, MAX_WALK_LENGTH).to(device)
model = GRFGP(X_train, Y_train, likelihood, kernel).to(device)

In [16]:
train_losses = train_model(model, likelihood, X_train, Y_train, lr=0.05, max_iter=1000, print_every=1)

Training:   0%|          | 0/1000 [00:00<?, ?it/s]

Iter 1/1000 - Loss: 1.4433
Iter 2/1000 - Loss: 1.4345
Iter 3/1000 - Loss: 1.4265
Iter 4/1000 - Loss: 1.4193
Iter 5/1000 - Loss: 1.4128
Iter 6/1000 - Loss: 1.4072
Iter 7/1000 - Loss: 1.4026
Iter 8/1000 - Loss: 1.3989
Iter 9/1000 - Loss: 1.3963
Iter 10/1000 - Loss: 1.3946
Iter 11/1000 - Loss: 1.3937
Iter 12/1000 - Loss: 1.3936
Iter 13/1000 - Loss: 1.3939
Iter 14/1000 - Loss: 1.3945
Iter 15/1000 - Loss: 1.3951
Iter 16/1000 - Loss: 1.3955
Iter 17/1000 - Loss: 1.3957
Iter 18/1000 - Loss: 1.3956
Iter 19/1000 - Loss: 1.3952
Iter 20/1000 - Loss: 1.3948
Iter 21/1000 - Loss: 1.3944
Iter 22/1000 - Loss: 1.3941
Iter 23/1000 - Loss: 1.3938
Iter 24/1000 - Loss: 1.3935
Iter 25/1000 - Loss: 1.3930
Iter 26/1000 - Loss: 1.3925
Iter 27/1000 - Loss: 1.3919
Iter 28/1000 - Loss: 1.3913
Iter 29/1000 - Loss: 1.3908
Iter 30/1000 - Loss: 1.3903
Iter 31/1000 - Loss: 1.3898
Iter 32/1000 - Loss: 1.3895
Iter 33/1000 - Loss: 1.3892
Iter 34/1000 - Loss: 1.3889
Iter 35/1000 - Loss: 1.3888
Iter 36/1000 - Loss: 1.3886
I

In [17]:
lml, rmse, nlpd = evaluate_model(model, likelihood, X_train, Y_train, X_test, Y_test, orig_std)

In [18]:
print(f"Log Marginal Likelihood: {lml:.4f}")
print(f"Test RMSE: {rmse:.4f}")
print(f"Test NLPD: {nlpd:.4f}")
print(f"kernel lengthscale: {kernel.beta.item():.4f}")
print(f"kernel stddev: {kernel.sigma_f.item():.4f}")

Log Marginal Likelihood: 254.3370
Test RMSE: 14.9204
Test NLPD: 99.2018
kernel lengthscale: 1.5613
kernel stddev: 1.3502
